In [4]:
import json
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

INPUT_FILE = "semantic_relevance_results.json"

# loading semantic relevance results
with open(INPUT_FILE, "r") as f:
    data = json.load(f)

df = pd.DataFrame(data)

print("\n✅ Loaded results")
print(df.head())

# -------- SUMMARY STATISTICS --------
print("\n📊 SUMMARY STATISTICS\n")

summary = {
    "Metric": ["Mean", "Median", "Std", "Min", "Max"],
    "Agg Similarity": [
        df["agg_similarity"].mean(),
        df["agg_similarity"].median(),
        df["agg_similarity"].std(),
        df["agg_similarity"].min(),
        df["agg_similarity"].max(),
    ],
    "Avg Keyword Similarity": [
        df["avg_keyword_similarity"].mean(),
        df["avg_keyword_similarity"].median(),
        df["avg_keyword_similarity"].std(),
        df["avg_keyword_similarity"].min(),
        df["avg_keyword_similarity"].max(),
    ]
}

summary_df = pd.DataFrame(summary)
print(summary_df)

# -------- HISTOGRAM: AGG SIMILARITY --------
plt.figure()
plt.hist(df["agg_similarity"], bins=50)
plt.xlabel("Semantic Similarity (Aggregated)")
plt.ylabel("Number of Datasets")
plt.title("Distribution of Keyword–Content Similarity")
plt.savefig("hist_agg_similarity.png")
plt.close()

# -------- HISTOGRAM: COMPARISON --------
plt.figure()
plt.hist(df["agg_similarity"], bins=50, alpha=0.5, label="Aggregated")
plt.hist(df["avg_keyword_similarity"], bins=50, alpha=0.5, label="Per-keyword avg")
plt.legend()
plt.xlabel("Similarity")
plt.ylabel("Frequency")
plt.title("Aggregated vs Per-keyword Similarity")
plt.savefig("hist_comparison.png")
plt.close()

# BOX PLOT
plt.figure()

plt.boxplot(df["agg_similarity"], vert=True)

median_val = df["agg_similarity"].median()

plt.text(
    1.05,
    median_val,
    f"Median: {median_val:.2f}",
    verticalalignment='center'
)

plt.ylabel("Semantic Similarity")
plt.title("Box Plot of Semantic Similarity Scores")

plt.savefig("boxplot_similarity.png", bbox_inches='tight')
plt.close()

# THRESHOLD ANALYSIS 
def categorize(score):
    if score >= 0.75:
        return "High"
    elif score >= 0.5:
        return "Moderate"
    else:
        return "Low"

df["category"] = df["agg_similarity"].apply(categorize)

category_counts = df["category"].value_counts(normalize=True) * 100

print("\n📊 THRESHOLD DISTRIBUTION (%)\n")
print(category_counts)

# -------- SIMILARITY RANGE ANALYSIS (0–1) --------
print("\n📊 SIMILARITY RANGE DISTRIBUTION (0–1)\n")

bins = [i/10 for i in range(11)]
labels = [f"{bins[i]:.1f}-{bins[i+1]:.1f}" for i in range(len(bins)-1)]

df["similarity_bin"] = pd.cut(
    df["agg_similarity"],
    bins=bins,
    labels=labels,
    include_lowest=True
)

bin_counts = df["similarity_bin"].value_counts().sort_index()
bin_percent = (bin_counts / len(df)) * 100

range_df = pd.DataFrame({
    "Range": bin_counts.index,
    "Count": bin_counts.values,
    "Percentage": bin_percent.values
})

print(range_df)

# -------- RANGE BAR PLOT --------
plt.figure()
plt.bar(range_df["Range"], range_df["Percentage"])
plt.xticks(rotation=45)
plt.xlabel("Similarity Range")
plt.ylabel("Percentage of Datasets")
plt.title("Distribution of Semantic Similarity (0–1 Range)")
plt.tight_layout()
plt.savefig("similarity_range_distribution.png")
plt.close()

# -------- SCATTER: KEYWORD COUNT VS SIMILARITY --------
plt.figure()
plt.scatter(df["num_keywords"], df["agg_similarity"])
plt.xlabel("Number of Keywords")
plt.ylabel("Semantic Similarity")
plt.title("Keyword Count vs Semantic Relevance")
plt.savefig("scatter_keywords_vs_similarity.png")
plt.close()

# -------- TOP & BOTTOM DATASETS --------
top = df.sort_values("agg_similarity", ascending=False).head(10)
bottom = df.sort_values("agg_similarity", ascending=True).head(10)

print("\n🏆 TOP 10 DATASETS\n")
print(top)

print("\n⚠️ BOTTOM 10 DATASETS\n")
print(bottom)

# -------- GAP ANALYSIS --------
df["gap"] = df["agg_similarity"] - df["avg_keyword_similarity"]

print("\n📉 GAP ANALYSIS\n")
print(df["gap"].describe())

# -------- CORRELATION --------
print("\n🔗 CORRELATION MATRIX\n")
correlation = df[["agg_similarity", "avg_keyword_similarity", "num_keywords"]].corr()
print(correlation)

# -------- SAVE OUTPUTS --------
summary_df.to_csv("summary_statistics.csv", index=False)
category_counts.to_csv("category_distribution.csv")
range_df.to_csv("similarity_range_distribution.csv", index=False)
top.to_csv("top_datasets.csv", index=False)
bottom.to_csv("bottom_datasets.csv", index=False)

print("\n💾 Files saved:")
print("- summary_statistics.csv")
print("- category_distribution.csv")
print("- similarity_range_distribution.csv")
print("- top_datasets.csv")
print("- bottom_datasets.csv")
print("- hist_agg_similarity.png")
print("- hist_comparison.png")
print("- boxplot_similarity.png")
print("- similarity_range_distribution.png")
print("- scatter_keywords_vs_similarity.png")

print("\n✅ Analysis complete!")


✅ Loaded results
                                          dataset_id  agg_similarity  \
0  http://data.europa.eu/88u/dataset/forestry-fac...        0.621063   
1  http://data.europa.eu/88u/dataset/operational-...        0.389611   
2  http://data.europa.eu/88u/dataset/2021-census-...        0.659039   
3  http://data.europa.eu/88u/dataset/register-of-...        0.776730   
4  http://data.europa.eu/88u/dataset/leaf-phenolo...        0.553954   

   avg_keyword_similarity  num_keywords  
0                0.261930             4  
1                0.143329             6  
2                0.448767             2  
3                0.587790             5  
4                0.528495             3  

📊 SUMMARY STATISTICS

   Metric  Agg Similarity  Avg Keyword Similarity
0    Mean        0.505355                0.286435
1  Median        0.529041                0.284125
2     Std        0.184874                0.104275
3     Min       -0.144203               -0.144203
4     Max        0.95079